In [2]:
import numpy as np
from gaddag import Gaddag, Node
import heapq
import random
import numpy as np
import tkinter as tk
import copy
import warnings

In [3]:
gaddag = Gaddag()

GADDAG loaded from gaddag.pkl


In [9]:
class Scrabble:
    def __init__(self, num_players: int=2, num_letters=7):
        """Initializes the Scrabble game

        Args:
            num_players (int, optional): Number of players in the game (2 <= n, n <= 4 ). Defaults to 2.
            num_letters (int, optional): Number of letters in each player's rack. Defaults to 7.

        Notes:
            Sets current player to 0
            Board is a 15x15 grid
        Raises:
            ValueError: Raises for invalid inputs (num_players, num_letters)
        """        
        # GAME SETUP ---------------------------------------------------
        # Number of letters in each player's rack
        if num_letters < 1:
            raise ValueError("Number of letters must be greater than 0.")
        self.num_letters = num_letters
        
        
        if num_players < 2 or num_players > 4:
            raise ValueError("Number of players must be between 2 and 4.")
        self.num_players = num_players

        # Letter counts in the bag
        self.letter_counts = list(
            "A" * 9
            + "B" * 2
            + "C" * 2
            + "D" * 4
            + "E" * 12
            + "F" * 2
            + "G" * 3
            + "H" * 2
            + "I" * 9
            + "J" * 1
            + "K" * 1
            + "L" * 4
            + "M" * 2
            + "N" * 6
            + "O" * 8
            + "P" * 2
            + "Q" * 1
            + "R" * 6
            + "S" * 4
            + "T" * 6
            + "U" * 4
            + "V" * 2
            + "W" * 2
            + "X" * 1
            + "Y" * 2
            + "Z" * 1
        )

        # Point values of each letter
        self.letter_values = {
            "A": 1, "B": 3, "C": 3, "D": 2, "E": 1, "F": 4, "G": 2, "H": 4, "I": 1,
            "J": 8, "K": 5, "L": 1, "M": 3, "N": 1, "O": 1, "P": 3, "Q": 10, "R": 1,
            "S": 1, "T": 1, "U": 1, "V": 4, "W": 4, "X": 8, "Y": 4, "Z": 10,
        }

        # Bonus square logic
        self.get_bonus_value = {
                1: lambda value, multiplier: (value * 2, multiplier),
                2: lambda value, multiplier: (value * 3, multiplier),
                3: lambda value, multiplier: (value, multiplier * 2),
                4: lambda value, multiplier: (value, multiplier * 3),
                5: lambda value, multiplier: (value, multiplier * 2),
            }

        self.bonus_squares = {1: "double letter", 2: "triple letter", 3: "double word", 4: "triple word"}
        self.square_colors = {
            0: "white",  # Normal square
            1: "lightblue",  # Double letter
            2: "indianred",  # Double letter
            3: "deepskyblue",  # Double word
            4: "red",  # Triple word
            5: "yellow", # center (also double word)
        }
        self.turn = 1

        # This stores all the prior word placements (i.e. board state)
        self.board = [
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0,0, 0, 0, 0, 0, 0],
            
        ]

        # Bonus square values
        # SEE self.bonus_squares
        self.bonuses = [
                      [ 4 , 0 , 0 , 1 , 0 , 0 , 0 , 4 , 0 , 0 , 0 , 1 , 0 , 0 , 4 ],
                      [ 0 , 3 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 3 , 0 ],
                      [ 0 , 0 , 3 , 0 , 0 , 0 , 1 , 0 , 1 , 0 , 0 , 0 , 3 , 0 , 0 ],
                      [ 1 , 0 , 0 , 3 , 0 , 0 , 0 , 1 , 0 , 0 , 0 , 3 , 0 , 0 , 1 ],
                      [ 0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 ],
                      [ 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 ],
                      [ 0 , 0 , 1 , 0 , 0 , 0 , 1 , 0 , 1 , 0 , 0 , 0 , 1 , 0 , 0 ],
                      [ 4 , 0 , 0 , 1 , 0 , 0 , 0 , 5 , 0 , 0 , 0 , 1 , 0 , 0 , 4 ],
                      [ 0 , 0 , 1 , 0 , 0 , 0 , 1 , 0 , 1 , 0 , 0 , 0 , 1 , 0 , 0 ],
                      [ 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 ],
                      [ 0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 , 0 , 3 , 0 , 0 , 0 , 0 ],
                      [ 1 , 0 , 0 , 3 , 0 , 0 , 0 , 1 , 0 , 0 , 0 , 3 , 0 , 0 , 1 ],
                      [ 0 , 0 , 3 , 0 , 0 , 0 , 1 , 0 , 1 , 0 , 0 , 0 , 3 , 0 , 0 ],
                      [ 0 , 3 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 2 , 0 , 0 , 0 , 3 , 0 ],
                      [ 4 , 0 , 0 , 1 , 0 , 0 , 0 , 4 , 0 , 0 , 0 , 1 , 0 , 0 , 4 ]
                      ]
        # ----------------------------------------------------------------

        self.players = {
            played_id: {
                "name": f"Player {played_id}",
                "score": 0,
                "rack": self.draw_letters(num_letters),
            }
            for played_id in range(num_players)
        }
        self.current_player = 0

    def get_game_state(self):
        """
        Returns:
            dict: The current game state. Keys: board, letter_counts, players, turn
        """        
        game_state = {
            "board": self.board,
            "letter_counts": self.letter_counts,
            "players": self.players,
            "turn": self.current_player,
        }
        return game_state

    def next_player(self):
        """
            Updates self.current_player to the next player
            
            Returns:
                int: The new current player
        """
        self.current_player = (self.current_player + 1) % self.num_players
        return self.current_player

    def draw_letters(self, n: int, isRemoval: bool=True, letter_counts=None) -> list[str]:
        """Draws letters from the bag

        Args:
            n (int): Number of letters to draw
            isRemoval (bool, optional): Whether to remove the drawn letters from the bag. Defaults to True
            letter_counts (list[str], optional): Dictionary of letter counts. Defaults to self.letter_counts.
            
        Returns:
            list[str]: List of drawn letters
        """
        
        # Sets default value of letter_counts pointer
        if letter_counts == None:
            letter_counts = self.letter_counts

        # Prevents drawing more letters than there are left
        n = min(n, len(letter_counts))

        letters = random.sample(letter_counts, n)

        # Removes the drawn letters from the letter_counts pointer
        if isRemoval:
            for letter in letters:
                letter_counts.remove(letter)

        return letters

    def display_board(self):
        """Displays the board using tkinter"""

        root = tk.Tk()

        canvas = tk.Canvas(root, width=600, height=600, bg="white")
        canvas.pack()

        cell_size = 40
        for i, row in enumerate(self.board):
            for j, tile in enumerate(row):
                x0, y0 = j * cell_size, i * cell_size
                x1, y1 = x0 + cell_size, y0 + cell_size

                # Determine the cell's background color
                if tile != 0:
                    color = "#D8B989"
                    width = 4
                else:
                    bonus = self.bonuses[i][j]
                    color = self.square_colors.get(bonus, "white")
                    width = 1

                canvas.create_rectangle(x0, y0, x1, y1, fill=color, outline="black", width=width)

                # Draw the letter tile or bonus text
                if tile != 0:
                    canvas.create_text(
                        x0 + cell_size / 2, 
                        y0 + cell_size / 2, 
                        text=tile.upper(), 
                        font=("Helvetica", 16, "bold"), 
                        fill="white"
                    )

                    # Draw the value in the bottom-right corner
                    value = self.letter_values.get(tile.upper(), 0)
                    canvas.create_text(
                        x1 - 5, 
                        y1 - 5, 
                        text=str(value), 
                        font=("Helvetica", 10, "bold"), 
                        anchor="se"
                    )
                elif self.bonuses[i][j] != 0:
                    bonus_text = {
                        1: "DL", 2: "TL", 3: "DW", 4: "TW", 5: "★"
                    }.get(self.bonuses[i][j], "")
                    canvas.create_text(
                        x0 + cell_size / 2, 
                        y0 + cell_size / 2, 
                        text=bonus_text, 
                        font=("Helvetica", 10), 
                        fill="black"
                    )

        root.mainloop()

    def print_board(self):
        for row in self.board:
            print(" ".join(str(cell) for cell in row))

    def play_word(self, word: str, score: str, start_position: tuple[int,int], is_transpose: bool):
        """Updates board, draws new letters, updates score, and switches turn

        Args:
            word (str): String of the word to play
            score (str): Points scored by the word
            start_position (tuple[int,int]): Absolute position of the first letter of the word (not transposed)
            is_transpose (bool): Whether the word is transposed

        Raises:
            ValueError: Raises when an illegal move is played
        """        
        x, y = start_position
        player = self.players[self.current_player]
        if is_transpose:
            x , y = y, x

        for letter in word:
            if self.board[y][x] != 0:
                pass
            else:
                if letter not in player["rack"]:
                    raise ValueError(f"Attempting to play an illegal move. Letter {letter} not in player {self.current_player}'s rack.")
                player["rack"].remove(letter)
                self.board[y][x] = letter

            if is_transpose:
                y += 1
            else:
                x += 1

        player["score"] += score

        num_letters_to_draw = self.num_letters - len(player["rack"])
        player["rack"] += self.draw_letters(n=num_letters_to_draw, isRemoval=True)

        self.next_player()

In [5]:
class Solver:
    def __init__(self, gaddag: Gaddag, game=Scrabble()):
        """Move Generator for scrabble

        Args:
            gaddag Gaddag: gaddag data structure of the dictionary you're using
                - SEE http://blog.wouterkoolen.info/Gaddag/post.html
        """

        self.gaddag = gaddag
        self.root = gaddag.root

        # Game state
        self.game = game
        self.board = self.game.board
        self.bonus_squares = self.game.bonus_squares
        self.rack_length = self.game.num_letters

    def is_word(self, word: str) -> bool:
        """Returns whether a word is in the dictionary

        Args:
            word (str): Word to check

        Returns:
            bool: is word in the dictionary?
        """

        node = self.root

        # checks the reversed path
        # ie. if the word is "RATE", it checks "ETAR#"

        for char in word[::-1] + "#":
            if char in node.children:
                node = node.children[char]
            else:
                return False

        return "$" in node.children

    def _get_candidate_letters(self, rack: list[str], board, position: tuple[int, int]):
        """
        Gets all letters that can be placed at a certain position given vertical constraints.

        Args:
            rack (list[str]): Characters in the rack.
            board (list[list[str]]): The current board state (based on transposition).
            position (tuple[int, int]): The position to check (based on transposition).

        Yields:
            str: Valid letters that can be placed at the position.
        """
        candidates = set()
        prefix = ""
        suffix = ""
        x, y = position

        if board[y][x] != 0:
            raise ValueError("Position is already occupied")

        # Get prefix
        y_1 = y - 1
        while y_1 >= 0 and board[y_1][x] != 0:
            prefix = board[y_1][x] + prefix
            y_1 -= 1

        # Get suffix
        y_2 = y + 1
        while y_2 < len(board) and board[y_2][x] != 0:
            suffix += board[y_2][x]
            y_2 += 1
            
        # Return rack if there are no vertical constraints
        if not prefix + suffix:
            return set(rack)

        # Checks if any letters in rack, complete the prefix-suffix pair
        for char in set(rack):
            if self.is_word(prefix + char + suffix):
                candidates.add(char)

        return candidates


    def get_tranposed_board(self, board=None):
        """
        Returns the transpose of the board
        """        
        if board is None:
            board = self.board
        return [list(row) for row in zip(*board)]

    def _build_valid_suffix(
        self, prefix: str, node: Node, rack: list[str], board, position: tuple[int, int], usedRack=False
    ) -> set:
        """Recursively builds all valid suffixes that can be built from a prefix and a rack and a position

        Args:
            prefix (str): A valid prefix
            node (Node): Node of the current last letter of the suffix
            rack (list[str]): All of the letters available to the player
            board (_type_): Current board state
            position (tuple[int, int]): The position to check
            valid_words (_type_, optional): All valid words given a prefix. Defaults to set().

        Returns:
            list[dict]: Has keys "word", "start_position", "score"
        """
        x, y = position
        valid_words = set()

        # While x,y is occupied, check if its part of a valid suffix
        # After this loop, x,y NEVER has a letter on it (either off the board or is empty)
        while x < len(board[0]) and board[y][x] != 0:

            if board[y][x] in node.children:
                node = node.children[board[y][x]]
                prefix += board[y][x]
                x += 1
            else:
                return valid_words

        # Checks for delimeter node (denotes a valid word)
        if "$" in node.children and usedRack:
            valid_words.add(prefix)

        # Recursive base cases (x out of bounds or no more letters to place)
        if x >= len(board[0]) or len(rack) == 0:
            return valid_words

        # Try all letters in the rack that are part of a valid suffix
        candidate_letters = self._get_candidate_letters(rack, board, (x, y))
        for char in candidate_letters:
            if char in node.children:
                # Create a new rack with the letter removed
                new_rack = rack.copy()
                new_rack.remove(char)

                # `usedRack` prevents returning a word where the player doesn't play a letter.
                usedRack = True
                valid_words.update(self._build_valid_suffix(prefix + char, node.children[char], new_rack, board, (x + 1, y), usedRack=usedRack))

        return valid_words

    def score_play(
        self, word: str, starting_position: tuple[int, int], board, transposed_board
    ) -> int:
        """Calculates the point score of a play (including all secondary words)

        Args:
            word (str): HORIZONTAL Collinear and adjacent letters played on the board
            starting_position (tuple[int, int]): The coordinates of the first letter of the word
            board (list[list[]]): board (relative to the tranpose of the word)
            transposed_board (list[list[]]): transposed board (relative to the tranpose of the word)

        Returns:
            int: The total score of the play
        """        

        # ----------------------------------------------
        def _score_word(word, starting_position, board) -> int:
            """Calculates the score of a word

            Args:
                word (str): String of the word to score
                starting_position (tuple(int, int)): x, y coordinates of the first letter of the word
                board (list[list[]]): board

            Returns:
                int: The score of the word
            """            
            multiplier = 1
            score = 0

            x, y = starting_position

            ## Below is a check for invalid plays
            # if not self.is_word(word):
            #     raise ValueError(f"{word} is not a valid word")

            for letter in word:
                square = self.game.bonuses[y][x]
                value = self.game.letter_values[letter]
                # checks the board that is passed in
                if square != 0 and board[y][x] == 0:
                    value, multiplier = self.game.get_bonus_value[square](
                        value, multiplier
                    )
                score += value
                x += 1
            score *= multiplier
            return score

        # ----------------------------------------------

        x, y = starting_position
        total_score = 0

        # This counter is for the 7 letter "bingo". If the player uses all 7 letters, they get a 50 point bonus
        counter = 0

        # This loop calls _score_word on all the secondary words
        for char in word:
            if board[y][x] != 0:
                x += 1
                continue
            counter += 1
            if (
                (y - 1) >= 0
                and board[y - 1][x] != 0
                or (y + 1) < len(board)
                and board[y + 1][x] != 0
            ):
                secondary_word = char
                y_1 = y - 1
                y_2 = y + 1
                while y_1 >= 0 and board[y_1][x] != 0:
                    secondary_word = board[y_1][x] + secondary_word
                    y_1 -= 1
                while y_2 < len(board) and board[y_2][x] != 0:
                    secondary_word += board[y_2][x]
                    y_2 += 1

                total_score += _score_word(
                    word=secondary_word,
                    starting_position=(y_1 + 1, x),
                    board=transposed_board,
                )
            x += 1

        # Add score of the main word
        total_score += _score_word(word=word, starting_position=starting_position, board=board)
        total_score += 50 if counter >= 7 else 0

        return total_score

    def _build_valid_prefix(
        self,
        suffix: str,
        node: Node,
        rack: list[str],
        board,
        position: tuple[int, int],
        valid_words: list = None,
        usedRack: bool = False,
    ) -> list:
        """Recursively builds a list of all valid prefixes that can be built from a rack and an origin


        ---------IMPORTANT---------
        - *** ALL LOGIC IS BASED ON HORIZONTAL WORD CONSTRUCTUION
        - You should only call this on positions s.t. there is no letter to the right of it. This avoids redudant outputs
        - Call this with the tranpose of board if you want the vertical prefixes
        --------------------------
        Args:

            suffix (str): Sub-Suffix to build prefixes from.
            node (Node): Node of the current first letter of suffix (ETA`R`, for prefix "RATE").
            rack (list[str]): Characters in the rack.
            board : Current board state.
            position (tuple[int, int]): The position to check.

        Returns:
            list[dict]: _description_
        """

        x, y = position

        if valid_words is None:
            valid_words = []

        # While x,y is occupied, check if its part of a valid prefix
        # After this loop, x,y NEVER has a letter on it (either off the board or is empty)
        while x >= 0 and board[y][x] != 0:
            if board[y][x] in node.children:
                node = node.children[board[y][x]]
                suffix = board[y][x] + suffix
                x -= 1
            else:
                return valid_words

        # Checks for delimeter node (denotes a valid prefix)
        if "#" in node.children:
            # The x value of this position denotes the square to the left of the prefix
            # This square is guaranteed to be empty
            new_position = (x + len(suffix) + 1, y)
            # Get the delimeter node
            # See gaddag strcuture for more info
            new_node = node.children["#"]

            # Save the starting position of the prefix
            start_position = (x + 1, y)

            words = self._build_valid_suffix(
                prefix=suffix,
                node=new_node,
                rack=rack,
                board=board,
                position=new_position,
                usedRack=usedRack,
            )
            for word in words:
                valid_words.append({"word": word, "start_position": start_position})

        # At this point, x,y is the square we are trying to place a new letter at
        # If this is out of bounds or we have no letters in the rack, return the valid prefixes
        if x < 0 or len(rack) == 0:
            return valid_words

        candidate_letters = self._get_candidate_letters(rack, board, (x, y))
        for char in candidate_letters:
            if char in node.children:
                # `place` the letter on the board and remove it from the rack
                new_rack = rack.copy()
                new_rack.remove(char)
                new_node = node.children[char]
                
                # Prevents returning a word where the player doesn't play a letter.
                usedRack = True
                valid_words = self._build_valid_prefix(
                    char + suffix, new_node, new_rack, board, (x - 1, y), valid_words, usedRack
                )

        return valid_words

    def _build_all_words(self, rack: list[str], num_words=16, board=None, first_move=False) -> list[dict]:
        """_summary_

        Args:
            rack (list[str]): letters avaible to player
            board (list[list[str]], optional): current board state. Defaults to self.game.board.
            num_words (int, optional): Number of words. Defaults to 20.
            first_move (bool, optional): If this is the first move of the game. Defaults to False.

        Notes:
            - Word building is done in two steps: PREFIX -> SUFFIX
            
            
        Returns:
            list[dict]: Sorted list of words and their scores
        """
        
        def _get_candidate_positions(board):
            for y in range(len(board)):
                for x in range(len(board[0])):
                    square = board[y][x]

                    hasLetter = square != 0
                    isOnRightBorder = x == (len(board) - 1)
                    hasEmptySquareToRight = (x+1 < len(board) and board[y][x+1] == 0)

                    hasLetterAbove = y - 1 >= 0 and board[y-1][x] != 0
                    hasLetterBelow = y + 1 < len(board) and board[y+1][x] != 0

                    if (hasLetter and (isOnRightBorder or hasEmptySquareToRight)) or (hasLetterAbove or hasLetterBelow):
                        if hasLetterAbove or hasLetterBelow:
                            yield (x, y)

        def _construct_words(board, transposed_board, rack, position, is_tranpose):
            """Builds all plays given a position on the board

            Args:
                board (list[list[]]): board
                transposed_board (list[list[]]): transposed board
                rack (list[str]): letters available to player
                position (tuple[int, int]): position to check
                is_tranpose (bool): is the board transposed?

            Returns:
                list[dict]: List of all valid words (each index has keys "word", "start_position", "score", "is_transpose")
            """            
            valid_words = []
            x, y = position
            words = self._build_valid_prefix(
                suffix="", node=self.root, rack=rack, board=board, position=(x, y)
            )

            # adding score / starting position to the horizontal words
            for play in words:
                word = play["word"]
                start_position = play["start_position"]
                score = self.score_play(
                    word=word,
                    starting_position=start_position,
                    board=board,
                    transposed_board=transposed_board,
                )
                valid_words.append(
                    {
                        "word": word,
                        "start_position": start_position,
                        "score": score,
                        "is_transpose": is_tranpose,
                    }
                )
            # ------------------------------------------------------
            return valid_words

        # ----------------------------------------------

        # Sets default value for board
        if board is None:
            board = self.game.board

        all_words = []

        transposed_board = self.get_tranposed_board(board=board)

        if first_move:
            # Only finds horizontal words for the first move (symmetry makes vertical words redudant)
            all_words += _construct_words(board, transposed_board, rack, (7, 7), False)

        else:
            # Finds all HORIZONTAL words
            for position in _get_candidate_positions(board):
                all_words += _construct_words(board, transposed_board, rack, position, False)

            # Finds all VERTICAL words
            for position in _get_candidate_positions(transposed_board):
                all_words += _construct_words(transposed_board, board, rack, position, True)

        # Sorts the words by score, returns the top {num_words}
        return heapq.nlargest(num_words, all_words, key=lambda x: x["score"])

In [6]:
class GameNode:
    def __init__(self, parent, game, solver, game_state, previous_move=None):
        """Game node including game state and children

        Args:
            parent (Game Node): Parent Node
            game (Scrabble): Scrabble game object
            game_state dict["board", "letter_counts", "players", "turn"]: Current game state
        """
        self.parent = parent
        self.game = game
        self.game_state = game_state
        self.previous_move = previous_move
        self.previous_move_score = (
            self.previous_move["score"] if self.previous_move else 0
        )
        self.solver = solver
        self.children = []


    def _update_board(self, move, board, rack):
        """Updates the game state after a move is played

        Args:
            move (dict): Move to play
            board (list[list[]]): board
            rack (list[str]): Player's rack
        """        
        # Play the move onto NEW BOARD
        word = move["word"]
        start_position = move["start_position"]
        is_transpose = move["is_transpose"]

        x, y = start_position
        if is_transpose:
            x, y = y, x
        for letter in word:
            if board[y][x] != 0:
                pass
            else:
                rack.remove(letter)
                board[y][x] = letter

            if is_transpose:
                y += 1
            else:
                x += 1

    def get_moves(self, isFirstMove=False, num_words=10):
        rack = self.game_state["players"][self.game_state["turn"]]["rack"]
        board = self.game_state["board"]

        # Gets top candidate moves from FindWords()
        moves = self.solver._build_all_words(rack=rack, board=board, first_move=isFirstMove, num_words=num_words)

        return moves

    def make_child(self, move):
        """Creates a child node of the current node

        Args:
            move (dict): The play made
            
        Notes:
            - CREATES A COPY OF THE GAME STATE
            - Updates the board, rack, and score for the new game state
            - Switches the turn
            - Draws new letters for the player
        """        
        new_game_state = copy.deepcopy(self.game_state)

        board = new_game_state["board"]
        turn = new_game_state["turn"]

        player = new_game_state["players"][turn]
        rack = player["rack"]

        self._update_board(move, board, rack)

        rack += self.game.draw_letters(
            self.game.num_letters - len(rack),
            isRemoval=True,
            letter_counts=new_game_state["letter_counts"],
        )
        
        player["score"] += move["score"]
        new_game_state["turn"] = (turn + 1) % self.game.num_players

        
        new_child = GameNode(self, self.game, self.solver, new_game_state, previous_move=move)
        self.children.append(new_child)


class ScrabbleAI:
    def __init__(self, game: Scrabble = Scrabble()):
        """Scrabble AI solver using Monte Carlo Tree Search

        Args:
            game (Scrabble, optional): Scrabble game object. Defaults to Scrabble().
        """        
        self.game = game
        self.solver = Solver(gaddag, game=game)

        game_state = {
            "board": game.board,
            "letter_counts": game.letter_counts,
            "players": game.players,
            "turn": game.current_player,
        }
        self.root = GameNode(
            None, game=self.game, solver=self.solver, game_state=game_state
        )

    def find_best_move(self, is_first_move=False, max_depth=3, width=10, num_words_explored=[3, 2, 1], random_opponent_rack: bool = True):
        """_summary_

        Args:
            is_first_move (bool, optional): If is the first move of the game. Defaults to False.
            max_depth (int, optional): Depth to explore game tree. Defaults to 10.
            width (int, optional): Number of random trials to run for each Node. Defaults to 3.
            num_words_explored (list[int], optional): Number of top words to explore at each node. Defaults to [3, 2, 1].
            random_opponent_rack (bool, optional): If false, simulations use the opponent's actual rack. Defaults to True.

        Notes:
            - The number of words explored should be the same as the max_depth
            - For best use, hyperparameters should follow the following properties:
                - num_words_explored should be generally decreasing
                    - num_words_explored[0] is the most important to keep large
                - width should be quite large, as it is the main source of randomness
                - a max_depth > 4 doesn't has very diminishing returns
                - random_opponent_rack: False, is akin to getting the true EV of a position 
            
        Returns:
            list[dict]: List of best moves and their expected values
        """       

        if max_depth != len(num_words_explored):
            
            if len(num_words_explored) > max_depth:
                num_words_explored = num_words_explored[:max_depth]
                warnings.warn(f"num_words_explore should have length {max_depth}. Trimming extra values. New num_words_explored = {num_words_explored}", UserWarning)
            else:
                num_words_explored += [num_words_explored[-1]] * (max_depth - len(num_words_explored))
                warnings.warn(f"num_words_explore should have length {max_depth}. Broadcasting empty values. New num_words_explored = {num_words_explored}", UserWarning)

        # Explore the tree to populate children
        self.explore_tree(
            is_first_move=is_first_move,
            max_depth=max_depth,
            width=width,
            num_words_explored=num_words_explored,
            random_opponent_rack=random_opponent_rack,
        )

        # Precompute EV values for all children
        best_moves = [
            {"move": move.previous_move, "ev": self._get_ev(move)}
            for move in self.root.children
        ]

        # Sort the moves by EV in descending order
        best_moves.sort(key=lambda x: x["ev"], reverse=True)

        return best_moves

    def _scale(self, x):
        # Min-max normalization
        min_val = np.min(x)
        max_val = np.max(x)
        range_val = max_val - min_val
        epsilon = 1e-10
        return (x - min_val) / (range_val + epsilon)

    def set_state(self, game_state):
        self.root = GameNode(None, game=self.game ,game_state=game_state, solver=self.solver)

    def _softmax(self, x, temperature=1):
        """Internal softmax function with temperature parameter"""
        e_x = np.exp((x - np.max(x)) / temperature)
        return e_x / e_x.sum()

    def _get_ev(self, node):
        """Internal function to recursively approximate the expected value of a move

        Is essentially a MCTS simulation step

        Args:
            node GameNode: current node

        Returns:
            np.float64: the expected value of a move
        """

        # recursive base case
        if not node.children:
            return node.previous_move_score

        # Get the expected value of the children
        values = [self._get_ev(child) for child in node.children]
        scaled_values = self._scale(values)
        probs = self._softmax(scaled_values, temperature=1)
        child_ev = np.dot(probs, values)
        
        # The move_ev is the difference between the score of the move you play, and the EV of the child nodes' moves
        move_ev = node.previous_move_score - child_ev
        return move_ev

    def explore_tree(
        self,
        is_first_move: bool,
        max_depth: int,
        random_opponent_rack: bool,
        num_words_explored: list[int],
        width: int,
    ):
        """Wrapper function to explore the game tree
        
        Args:
            *** SEE ScrabbleAI.find_best_move *** 
        
        Notes:
            - There is almost no reason to include previously calculated EVs in the tree
            - After playing a move and getting a new rack, the game tree has shrunk by a factor of about 4 digits
        """        
        def _explore(node, num_words_explored: int, current_depth: int):
            if current_depth >= max_depth:
                return

            candidate_words = node.get_moves(num_words=num_words_explored[current_depth])

            # Explore each move and make it a child node
            for move in candidate_words:
                node.make_child(move)
                _explore(node.children[-1], num_words_explored=num_words_explored, current_depth=current_depth + 1)  # Recursively explore child node

        # Clear children from previous exploration
        self.root.children = []
        candidate_words = self.root.get_moves(isFirstMove=is_first_move, num_words=num_words_explored[0])

        for move in candidate_words:
            self.root.make_child(move)

        # Width denotes the number of random trials ran for each node
        for i in range(width):
            for child in self.root.children:

                if random_opponent_rack:
                    child.game_state["players"][1]["rack"] = self.game.draw_letters(
                        self.game.num_letters, isRemoval=True, letter_counts=child.game_state["letter_counts"]
                    )
                else:
                    child.game_state["players"][1]["rack"] = self.game.players[1]["rack"]
                _explore(child, num_words_explored, 1)

        return candidate_words 

In [10]:
game = Scrabble()
ai = ScrabbleAI(game)

In [12]:
move_idx = 0
while True:
    
    moves = ai.find_best_move(is_first_move=(move_idx==0), random_opponent_rack=True, max_depth=1, width=10, num_words_explored=[3, 2, 1])
    if not moves:
        break
    
    best_move = moves[0]["move"]
    
    game.play_word(
        word=best_move["word"], score=best_move["score"], start_position=best_move["start_position"], is_transpose=best_move["is_transpose"]
    )
    
    ai.set_state(game.get_game_state())
    
    move_idx += 1


C:\Users\luequ\AppData\Local\Temp\ipykernel_28712\4064005303.py:141: UserWarning: num_words_explore should have length 1. Trimming extra values. New num_words_explored = [3]
  warnings.warn(f"num_words_explore should have length {max_depth}. Trimming extra values. New num_words_explored = {num_words_explored}", UserWarning)


In [ ]:
from scrabble import Scrabble as THING
from scrabble_ai import ScrabbleAI as THING2


def simulate_game(print_board=True, display_board=False):
    game = Scrabble()
    ai = ScrabbleAI()

    move_idx = 0
    while True:
        
        moves = ai.find_best_move(is_first_move=(move_idx==0), random_opponent_rack=True, max_depth=3, width=10, num_words_explored=[3, 2, 1])
        if not moves:
            break
        
        best_move = moves[0]["move"]
        
        game.play_word(
            word=best_move["word"], score=best_move["score"], start_position=best_move["start_position"], is_transpose=best_move["is_transpose"]
        )
        
        game.print_board()        
        
        ai.set_state(game.get_game_state())
        
        move_idx += 1

    state = game.get_game_state()
    players = state["players"]
    for player in players:
        print(f"{player['name']}: {player['score']}")
    game.display_board()

if __name__ == "__main__":
    simulate_game(print_board=True, display_board=False)


In [14]:
state  = game.get_game_state()


In [15]:
state

{'board': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 'G', 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 'A', 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 'B', 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 'A', 'Y', 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 'C', 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 'E', 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 'V', 'E', 'X', 'T', 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 'O', 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 'U', 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 'S', 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
 'letter_counts': ['A',
  'A',
  'A',
  'A',
  'A',
  'A',
  'B',
  'C',
  'D',
  'D',
  'D',
  'D',
  'E',
  'E',
  'E',
  'E',
  'E',
  'E',
  'E',
  'E',
  'E',
  'F',
  'F',
  'G',
  'G',
  'H',
  'I',
  'I',
  'I',
  'I',


Finished Running Simulation:
----------------------------
Player 0: 56
Player 1: 81
